# RQ4 — Showcase Evaluation & Calibration

**Research Question**: Does the meta-learner generalize to completely unseen datasets?

Evaluates the trained meta-learner on 9 held-out showcase datasets that were
never part of meta-training. For each dataset:
1. Extract Option A meta-features (with median imputation for any NaN)
2. Predict best clustering method (classifier)
3. Predict expected LSE per method (regressor)
4. Run the predicted method, oracle, and k-means baseline
5. Compare true LSE for each strategy

Also produces **calibration plots** (predicted vs actual LSE per method) to
empirically determine the confidence floor, replacing the arbitrary 0.60 from CLAUDE.md.

**Outputs**: `outputs/figures/showcase_comparison.csv`, `lse_calibration.png`

In [7]:
import os, sys, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

ROOT        = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR     = os.path.join(ROOT, 'data', 'raw')
MODELS_DIR  = os.path.join(ROOT, 'outputs', 'models')
FIGURES_DIR = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(ROOT, 'src'))
openml.config.cache_directory = RAW_DIR

SEED = 42
np.random.seed(SEED)
print('Paths OK')

Paths OK


In [8]:
from lse import compute_lse, groundtruth_accuracy
from clustering import (
    pseudo_kmeans, pseudo_dbscan, pseudo_agglomerative,
    pseudo_gmm, pseudo_autoencoder, pseudo_dictlearn,
)
from metafeatures import extract_optA
print('Modules loaded')

Modules loaded


In [9]:
SHOWCASE_DATASETS = [
    {'id': 61,    'name': 'Iris'},
    {'id': 187,   'name': 'Wine'},
    {'id': 15,    'name': 'Breast Cancer Wisconsin'},
    {'id': 53,    'name': 'Heart Disease (UCI)'},
    {'id': 40966, 'name': 'Palmer Penguins'},
    {'id': 37,    'name': 'Diabetes (Pima)'},
    {'id': 54,    'name': 'Vehicle Silhouettes'},
    {'id': 1590,  'name': 'Adult Income'},
    {'id': 1597,  'name': 'Credit Card Fraud'},
]

METHODS = {
    'kmeans'   : pseudo_kmeans,
    'dbscan'   : pseudo_dbscan,
    'agg'      : pseudo_agglomerative,
    'gmm'      : pseudo_gmm,
    'autoenc'  : pseudo_autoencoder,
    'dictlearn': pseudo_dictlearn,
}
METHOD_NAMES = list(METHODS.keys())
print(f'{len(SHOWCASE_DATASETS)} showcase datasets')

9 showcase datasets


In [10]:
with open(os.path.join(MODELS_DIR, 'meta_clf_optA.pkl'), 'rb') as f:
    clf_bundle = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'meta_reg_optA.pkl'), 'rb') as f:
    reg_bundle = pickle.load(f)

meta_clf      = clf_bundle['pipeline']
clf_feat_cols = clf_bundle['feature_cols']
meta_reg      = reg_bundle['pipeline']
reg_feat_cols = reg_bundle['feature_cols']
lse_cols      = reg_bundle['lse_cols']

print(f'Classifier: {len(clf_feat_cols)} features')
print(f'Regressor : {len(reg_feat_cols)} features')
print(f'LSE cols  : {lse_cols}')

Classifier: 20 features
Regressor : 20 features
LSE cols  : ['LSE_kmeans', 'LSE_dbscan', 'LSE_agg', 'LSE_gmm', 'LSE_autoenc', 'LSE_dictlearn']


In [11]:
# Imputer for showcase datasets that may have NaN
# (showcase datasets are not filtered for missing values the way meta-training is)
_imputer = SimpleImputer(strategy='median')


def load_and_split(dataset_id):
    ds = openml.datasets.get_dataset(
        dataset_id, download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )
    X = X.select_dtypes(include=[np.number]).astype(float)
    # Impute NaN in showcase datasets before any computation
    if X.isnull().any().any():
        X = pd.DataFrame(
            SimpleImputer(strategy='median').fit_transform(X),
            columns=X.columns
        )
    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))
    X_tr, X_te, y_tr, y_te = train_test_split(
        X.values, y_enc, test_size=0.2,
        random_state=SEED, stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te


def scale(X_tr, X_te):
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_te)

In [12]:
records = []

for ds_info in SHOWCASE_DATASETS:
    did  = ds_info['id']
    name = ds_info['name']
    print(f'\n── {name} (id={did}) ──')

    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        if X_tr.shape[1] == 0:
            print('  SKIP: no numeric features')
            continue

        X_tr_sc, X_te_sc = scale(X_tr, X_te)
        n_cls = len(np.unique(y_tr))
        gt_acc = groundtruth_accuracy(X_tr, y_tr, X_te, y_te)
        print(f'  balanced_gt={gt_acc:.3f}  n_cls={n_cls}  n_tr={len(X_tr)}')

        # Step 1: meta-features (Option A; label-free, computed on scaled train data)
        feats = extract_optA(X_tr_sc, n_cls)
        feat_vec_clf = np.array([[feats.get(c, np.nan) for c in clf_feat_cols]])
        feat_vec_reg = np.array([[feats.get(c, np.nan) for c in reg_feat_cols]])

        # Step 2: classify
        predicted_method = meta_clf.predict(feat_vec_clf)[0]
        print(f'  Predicted best method: {predicted_method}')

        # Step 3: regress
        predicted_lse_vec = meta_reg.predict(feat_vec_reg)[0]
        predicted_lse = dict(zip(
            [c.replace('LSE_', '') for c in lse_cols],
            predicted_lse_vec,
        ))
        print(f'  Predicted LSE: ' +
              '  '.join(f'{m}={v:.3f}' for m, v in predicted_lse.items()))

        # Step 4: run all 6 methods
        true_lse = {}
        for mname, fn in METHODS.items():
            try:
                pseudo = fn(X_tr_sc, n_cls)
                lse_val, _ = compute_lse(
                    X_tr, y_tr, X_te, y_te, pseudo, gt_acc,
                    method_name=mname, dataset_name=name, verbose=False,
                )
                true_lse[mname] = round(lse_val, 4)
            except Exception as e:
                print(f'    {mname} FAILED: {e}')
                true_lse[mname] = np.nan

        # Step 5: outcomes
        valid_methods = {m: v for m, v in true_lse.items() if not np.isnan(v)}
        if not valid_methods:
            print('  All methods failed — skipping')
            continue

        oracle_method = max(valid_methods, key=valid_methods.get)
        oracle_lse_actual = valid_methods[oracle_method]
        pred_lse_actual   = true_lse.get(predicted_method, np.nan)
        kmeans_lse        = true_lse.get('kmeans', np.nan)

        print(f'  Oracle  : {oracle_method} → LSE={oracle_lse_actual:.3f}')
        print(f'  Predicted ({predicted_method}): LSE={pred_lse_actual:.3f}')
        print(f'  k-means baseline: LSE={kmeans_lse:.3f}')

        rec = {
            'dataset'            : name,
            'dataset_id'         : did,
            'n_classes'          : n_cls,
            'gt_acc'             : round(gt_acc, 3),
            'predicted_method'   : predicted_method,
            'predicted_lse_actual': round(pred_lse_actual, 3) if not np.isnan(pred_lse_actual) else np.nan,
            'oracle_method'      : oracle_method,
            'oracle_lse'         : round(oracle_lse_actual, 3),
            'kmeans_lse'         : round(kmeans_lse, 3) if not np.isnan(kmeans_lse) else np.nan,
            'hit'                : predicted_method == oracle_method,
            'gap_vs_oracle'      : round(oracle_lse_actual - pred_lse_actual, 3) if not np.isnan(pred_lse_actual) else np.nan,
            'gain_vs_kmeans'     : round(pred_lse_actual - kmeans_lse, 3) if not np.isnan(pred_lse_actual) else np.nan,
        }
        for m, v in true_lse.items():
            rec[f'true_lse_{m}'] = v
        for m, v in predicted_lse.items():
            rec[f'pred_lse_{m}'] = round(v, 4)

        records.append(rec)

    except Exception as e:
        print(f'  FAILED: {e}')

print('\nDone.')


── Iris (id=61) ──
  balanced_gt=0.900  n_cls=3  n_tr=120
  Predicted best method: gmm
  Predicted LSE: kmeans=0.664  dbscan=0.550  agg=0.648  gmm=0.697  autoenc=0.669  dictlearn=0.537
  Oracle  : gmm → LSE=1.037
  Predicted (gmm): LSE=1.037
  k-means baseline: LSE=0.889

── Wine (id=187) ──
  balanced_gt=1.000  n_cls=3  n_tr=142
  Predicted best method: gmm
  Predicted LSE: kmeans=0.747  dbscan=0.642  agg=0.698  gmm=0.794  autoenc=0.759  dictlearn=0.617
  Oracle  : kmeans → LSE=1.000
  Predicted (gmm): LSE=1.000
  k-means baseline: LSE=1.000

── Breast Cancer Wisconsin (id=15) ──
  balanced_gt=0.952  n_cls=2  n_tr=559
  Predicted best method: kmeans
  Predicted LSE: kmeans=0.837  dbscan=0.717  agg=0.855  gmm=0.871  autoenc=0.844  dictlearn=0.698
  Oracle  : agg → LSE=1.000
  Predicted (kmeans): LSE=0.989
  k-means baseline: LSE=0.989

── Heart Disease (UCI) (id=53) ──
  balanced_gt=0.817  n_cls=2  n_tr=216
  Predicted best method: autoenc
  Predicted LSE: kmeans=0.843  dbscan=0.660  

In [13]:
results_df = pd.DataFrame(records)

SUMMARY_COLS = [
    'dataset', 'n_classes', 'gt_acc',
    'predicted_method', 'predicted_lse_actual',
    'oracle_method', 'oracle_lse', 'kmeans_lse',
    'hit', 'gap_vs_oracle', 'gain_vs_kmeans',
]
summary = results_df[SUMMARY_COLS].copy()
print(summary.to_string(index=False))

n = len(summary)
print(f'\nTop-1 accuracy        : {summary["hit"].mean():.1%}  ({summary["hit"].sum()}/{n})')
print(f'Mean gap vs oracle    : {summary["gap_vs_oracle"].mean():.3f}')
print(f'Mean gain vs k-means  : {summary["gain_vs_kmeans"].mean():.3f}')

OUT_PATH = os.path.join(FIGURES_DIR, 'showcase_comparison.csv')
results_df.to_csv(OUT_PATH, index=False)
print(f'\nSaved → {OUT_PATH}')

                dataset  n_classes  gt_acc predicted_method  predicted_lse_actual oracle_method  oracle_lse  kmeans_lse   hit  gap_vs_oracle  gain_vs_kmeans
                   Iris          3   0.900              gmm                 1.037           gmm       1.037       0.889  True          0.000           0.148
                   Wine          3   1.000              gmm                 1.000        kmeans       1.000       1.000 False          0.000           0.000
Breast Cancer Wisconsin          2   0.952           kmeans                 0.989           agg       1.000       0.989 False          0.010           0.000
    Heart Disease (UCI)          2   0.817          autoenc                 1.000       autoenc       1.000       0.990  True          0.000           0.010
        Palmer Penguins          8   1.000           kmeans                 0.252           agg       0.391       0.252 False          0.139           0.000
        Diabetes (Pima)          2   0.741              gm

## Calibration Analysis

For each method, plot predicted LSE (from meta-regressor) vs actual LSE on showcase datasets.
This reveals whether the regressor is well-calibrated and helps choose an empirical
confidence floor to replace the arbitrary 0.60 in CLAUDE.md.

In [14]:
if len(results_df) < 3:
    print('Too few showcase results for calibration analysis.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    axes = axes.ravel()
    cal_errors = {}

    for ci, m in enumerate(['kmeans', 'dbscan', 'agg', 'gmm', 'autoenc', 'dictlearn']):
        true_col = f'true_lse_{m}'
        pred_col = f'pred_lse_{m}'
        if true_col not in results_df or pred_col not in results_df:
            continue

        valid = results_df[[true_col, pred_col]].dropna()
        if len(valid) == 0:
            continue

        true_vals = valid[true_col].values
        pred_vals = valid[pred_col].values
        mae = float(np.abs(true_vals - pred_vals).mean())
        cal_errors[m] = mae

        ax = axes[ci]
        ax.scatter(pred_vals, true_vals, s=60, alpha=0.7, color='steelblue',
                   edgecolors='white', linewidths=0.5)
        lim = [min(pred_vals.min(), true_vals.min()) - 0.05,
               max(pred_vals.max(), true_vals.max()) + 0.05]
        ax.plot(lim, lim, 'k--', lw=1, alpha=0.5, label='Perfect calibration')
        ax.set_xlabel('Predicted LSE')
        ax.set_ylabel('Actual LSE')
        ax.set_title(f'{m}  (MAE={mae:.3f})')
        ax.set_xlim(lim)
        ax.set_ylim(lim)

    plt.suptitle('LSE Calibration: Predicted vs Actual per Method (Showcase datasets)',
                 fontsize=12)
    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, 'lse_calibration.png')
    fig.savefig(path, dpi=130)
    plt.close()
    print(f'Saved → {path}')

    print('\n=== Calibration MAE per method ===')
    for m, mae in sorted(cal_errors.items(), key=lambda x: x[1]):
        print(f'  {m:10s}  MAE={mae:.4f}')
    overall_cal_mae = float(np.mean(list(cal_errors.values())))
    print(f'  Overall: MAE={overall_cal_mae:.4f}')

    # Empirical confidence floor
    # If predicted LSE < floor, the prediction is unreliable
    pred_best_vals = results_df.apply(
        lambda row: row.get(f'pred_lse_{row["predicted_method"]}', np.nan), axis=1
    ).dropna()
    actual_best_vals = results_df['predicted_lse_actual'].dropna()

    overoptimistic = pred_best_vals - actual_best_vals
    print(f'\n=== Overoptimism in predicted best-method LSE ===')
    print(f'  Mean overestimate: {overoptimistic.mean():+.4f}')
    print(f'  Std: {overoptimistic.std():.4f}')
    print(f'  Suggested confidence floor: predicted LSE > {(actual_best_vals.mean() - actual_best_vals.std()):.2f}')
    print('  (replace the hardcoded 0.60 in CLAUDE.md with this empirical value)')

Saved → c:\MLResearch\outputs\figures\lse_calibration.png

=== Calibration MAE per method ===
  dictlearn   MAE=0.0733
  dbscan      MAE=0.1065
  agg         MAE=0.1407
  gmm         MAE=0.1432
  autoenc     MAE=0.1448
  kmeans      MAE=0.1649
  Overall: MAE=0.1289

=== Overoptimism in predicted best-method LSE ===
  Mean overestimate: -0.0662
  Std: 0.2089
  Suggested confidence floor: predicted LSE > 0.52
  (replace the hardcoded 0.60 in CLAUDE.md with this empirical value)


In [15]:
print('=' * 60)
print('RQ4 SHOWCASE EVALUATION SUMMARY')
print('=' * 60)
print(f'  Showcase datasets evaluated : {len(results_df)}')
if len(results_df) > 0:
    hits = results_df['hit'].sum()
    print(f'  Top-1 accuracy              : {hits}/{len(results_df)} = {hits/len(results_df):.1%}')
    print(f'  Mean gap vs oracle          : {results_df["gap_vs_oracle"].mean():.3f} LSE')
    print(f'  Mean gain vs k-means        : {results_df["gain_vs_kmeans"].mean():.3f} LSE')
print()
print('Figures saved to outputs/figures/')
print('Phase complete.')

RQ4 SHOWCASE EVALUATION SUMMARY
  Showcase datasets evaluated : 8
  Top-1 accuracy              : 3/8 = 37.5%
  Mean gap vs oracle          : 0.051 LSE
  Mean gain vs k-means        : -0.013 LSE

Figures saved to outputs/figures/
Phase complete.
